Notebook de exploracion. La logica final va en `src/`.

## Qué es yelp

Yelp es una plataforma estadounidense (2004) donde los usuarios reseñan y puntúan negocios locales (restaurantes, tiendas, servicios...), consultan info del negocio (horario, ubicación, atributos como "acepta reservas" o "tiene terraza"), dejan tips cortos, hacen check-in cuando visitan un sitio, y se conectan entre ellos como amigos. Se monetiza sobre todo con publicidad/listados destacados para negocios.

In [1]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "common").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("No se encontró la carpeta 'common'.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from common.spark_session import create_spark_session

# 1. Data Loading

Reutilizo `create_spark_session` de `common/spark_session.py` (mismo patrón que en los proyectos anteriores) y cargo `business.json` y `checkin.json` como DataFrames de Spark.

In [3]:
spark = create_spark_session(app_name="yelp-nested-json")
spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Parallelism:", spark.sparkContext.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/07 14:08:45 WARN Utils: Your hostname, MacBook-Air-de-Gloria.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.20 instead (on interface en0)
26/09/07 14:08:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/gloriadelriomarquez/Documents/Career/spark-lab/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/07 14:09:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.2.0
Master: local[*]
Parallelism: 8


In [4]:
DATASETS_PATH = PROJECT_ROOT / "datasets" / "03-yelp" / "Yelp JSON"

for file in DATASETS_PATH.iterdir():
    print(file.name)

Yelp Dataset Documentation & ToS copy.pdf
yelp_dataset.tar
yelp_academic_dataset_checkin.json
Dataset_User_Agreement.pdf
yelp_academic_dataset_tip.json
yelp_academic_dataset_review.json
yelp_academic_dataset_business.json
yelp_academic_dataset_user.json


In [5]:
dfs = {}
for file in DATASETS_PATH.iterdir():
    if file.name.startswith('yelp_academic_dataset_'): 
        name = file.name.removeprefix("yelp_academic_dataset_").removesuffix(".json")
        dfs[name] = spark.read.json(str(file))

for name, df in dfs.items():
    print(f"=== {name} ===")
    df.printSchema()

=== checkin ===
root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)

=== tip ===
root
 |-- business_id: string (nullable = true)
 |-- compliment_count: long (nullable = true)
 |-- date: string (nullable = true)
 |-- text: string (nullable = true)
 |-- user_id: string (nullable = true)

=== review ===
root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)

=== business ===
root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullab

## Alcance de este proyecto

Nos centramos en `business.json` (el núcleo, con JSON anidado real: `attributes`, `hours`, `categories`) y `checkin.json` (de apoyo, mismo tipo de técnica — `explode` sobre una lista). Dejamos fuera `review.json` y `user.json`: son tablas planas y muy pesadas (5.3 GB y 3.3 GB), no aportan una técnica de anidado distinta a lo que ya practicamos en el proyecto 01.

In [6]:
business = spark.read.json(str(DATASETS_PATH / "yelp_academic_dataset_business.json"))
checkin = spark.read.json(str(DATASETS_PATH / "yelp_academic_dataset_checkin.json"))

# 2. Data Inventory

Antes de mirar el contenido, un vistazo rápido al tamaño de cada tabla: número de filas y columnas.

In [7]:
# nº de filas y columnas de cada tabla

print(f"business: {business.count()} filas, {len(business.columns)} columnas")
print(f"checkin: {checkin.count()} filas, {len(checkin.columns)} columnas")

business: 150346 filas, 14 columnas


checkin: 131930 filas, 2 columnas


# 3. Data Structure

Reviso el schema de cada dataset para identificar qué columnas son anidadas (struct/array) y cuáles son planas, antes de decidir cómo aplanarlas.

In [8]:
business.printSchema()

root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: string (nullable = true)
 |    |-- GoodForDancing: str

In [9]:
business.show(1)

+--------------------+--------------------+--------------------+--------------------+-------------+-----+-------+----------+------------+--------------------+-----------+------------+-----+-----+
|             address|          attributes|         business_id|          categories|         city|hours|is_open|  latitude|   longitude|                name|postal_code|review_count|stars|state|
+--------------------+--------------------+--------------------+--------------------+-------------+-----+-------+----------+------------+--------------------+-----------+------------+-----+-----+
|1616 Chapala St, ...|{NULL, NULL, NULL...|Pns2l4eNsfO8kk83d...|Doctors, Traditio...|Santa Barbara| NULL|      0|34.4266787|-119.7111968|Abby Rappoport, L...|      93101|           7|  5.0|   CA|
+--------------------+--------------------+--------------------+--------------------+-------------+-----+-------+----------+------------+--------------------+-----------+------------+-----+-----+
only showing top 1 r

In [10]:
#checkin
checkin.printSchema()


root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)



In [11]:
checkin.show(1, truncate=False)

+----------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|business_id           |date                                                                                                                                                                                                                                 |
+----------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|---kPU91CF4Lq2-WlRu9Lw|2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020-07-24 22:42:27, 2020-10-24 21:36:13, 2020-12-09 21:23:33, 2021-01-20 17:34:57, 2021-04-30 21:02:03, 2021-05-25 21:16:54, 2021-08-06 21:08:08, 2021-10-02 15:15:42, 2

> **Observación:**
>
> `attributes` es un **struct** con un campo fijo y nombrado por cada atributo posible
> (`AcceptsInsurance`, `Alcohol`, `Ambience`, `BikeParking`, `BusinessParking`, `GoodForKids`...),
> todos de tipo `string` y nulos si el negocio no especificó ese atributo — no es un `map`.
>
> `hours` es igual: un **struct** con un campo fijo por cada día de la semana (`Monday`...`Sunday`),
> también `string` (ej. `"8:0-22:0"`) y nulo si el negocio no publicó horario ese día.
>
> Como los dos son structs de esquema fijo (no `map`), **no hace falta `explode()`** para
> aplanarlos: se accede a cada campo con notación de punto (`attributes.GoodForKids`,
> `hours.Monday`...) y se seleccionan como columnas nuevas los que interesen. `explode()` solo
> hace falta para `categories` y `checkin.date`, que sí son listas/strings de longitud variable.

# 4. Data Understanding

Estadísticas básicas de las columnas numéricas (`stars`, `review_count`) y cardinalidad de las categóricas (`state`, `city`, `is_open`...), para entender qué tipo de variable es cada una antes de aplanar nada.

In [12]:
from common.profiling import numeric_summary, quantile_summary

# numeric_cols_summary 
numeric_summary(business).show()

+-------+------------------+------------------+------------------+------------------+------------------+
|summary|           is_open|          latitude|         longitude|      review_count|             stars|
+-------+------------------+------------------+------------------+------------------+------------------+
|  count|            150346|            150346|            150346|            150346|            150346|
|   mean|0.7961502135075094|36.671150064145664|-89.35733948971436| 44.86656113232144|3.5967235576603303|
| stddev|0.4028599390900638| 5.872758917014048|14.918501679930612|121.12013570117087|0.9744207509201349|
|    min|                 0|         27.555127|       -120.095137|                 5|               1.0|
|    25%|                 1|        32.1871049|        -90.357825|                 8|               3.0|
|    50%|                 1|         38.777262|       -86.1212182|                15|               3.5|
|    75%|                 1|        39.9540239|    -75.

In [13]:
quantile_summary(business, ["review_count"]).show()

+------------+---+---+------+----+----+-----+-----+------+
|    variable|min|p25|median| p75| p90|  p95|  p99|  p999|
+------------+---+---+------+----+----+-----+-----+------+
|review_count|5.0|8.0|  15.0|37.0|96.0|171.0|449.0|7568.0|
+------------+---+---+------+----+----+-----+-----+------+



In [14]:
from common.profiling import cardinality_profile

# cardinality_profile sobre las columnas categóricas de business (state, city, is_open...)
cardinality_profile(business).show()

+------------+---------------+---------------+
| column_name|distinct_values|cardinality_pct|
+------------+---------------+---------------+
| business_id|         150346|          100.0|
|    latitude|         135593|          90.19|
|   longitude|         131918|          87.74|
|     address|         122844|          81.71|
|        name|         114117|           75.9|
|  categories|          83160|          55.31|
|  attributes|          67212|           44.7|
|       hours|          49822|          33.14|
| postal_code|           3362|           2.24|
|        city|           1416|           0.94|
|review_count|           1158|           0.77|
|       state|             27|           0.02|
|       stars|              9|           0.01|
|     is_open|              2|            0.0|
+------------+---------------+---------------+



## Conclusiones

### Variables categóricas

- `state` (27 valores) — estado de EEUU/Canadá del negocio.
- `city` (1.416 valores) — más granular que `state`.
- `postal_code` (3.362 valores) — nivel más fino todavía.
- `is_open` (2 valores) — binaria, negocio abierto/cerrado.
- `stars` (9 valores: 1.0, 1.5, 2.0 ... 5.0) — categórica ordinal, no continua (solo pasos de 0.5).

### Variables numéricas

- `review_count` — muy sesgada a la derecha (mediana 15, p99 449, máximo 7.568): mejor usar
  mediana que media al resumir, y el mínimo de 5 no es un filtro nuestro, viene así del propio
  dataset público de Yelp (solo incluye negocios con al menos 5 reseñas).
- `latitude` / `longitude` — numéricas pero de uso posicional (mapas/distancias), no como métrica
  de negocio en sí.

### Variables anidadas (a tratar en Feature Engineering)

- `categories` — string separado por comas. Su 55,31% de valores distintos no es cardinalidad
  categórica real: cada negocio tiene su propia combinación de categorías, confirma que hay que
  trocearla (`split` + `explode`) antes de poder analizarla por categoría individual.
- `attributes` — struct de esquema fijo (confirmado en la sección 3, no `map`), ~40 campos string.
- `hours` — struct de esquema fijo (confirmado en la sección 3, no `map`), un campo string por
  día de la semana.

### Observaciones

- `business_id` es 100% único, como corresponde a una clave primaria.
- `latitude`, `longitude`, `address` y `name` tienen cardinalidad muy alta (76-90%) — son casi
  identificadores, no sirven como variables categóricas para agrupar.
- El alto % de valores distintos en `attributes` tampoco es cardinalidad "real": al ser un
  struct, `countDistinct` compara la combinación completa de los ~40 subcampos a la vez, así que
  sale alto aunque cada subcampo individual tenga pocos valores posibles (`True`/`False`/`None`,
  etc.) — otra razón más para aplanarlo antes de sacar conclusiones sobre atributos concretos.


# 5. Data Quality Assessment

Reviso duplicados y valores nulos antes de tocar nada. Ojo con los nulos en `attributes`/`hours`/`categories`: igual que en el proyecto de NYC Taxi los nulos de `payment_type = Flex Fare` no eran un error sino un tipo de registro distinto, aquí un negocio sin `hours` probablemente significa "no publicó su horario", no un fallo de carga. Antes de "limpiarlo", compruebo si el patrón de nulos está relacionado con algo (¿son negocios cerrados, `is_open = 0`? ¿de una categoría concreta?).

In [15]:
from common.profiling import duplicate_count

# comprobación genreal de filas duplicadas 
duplicate_count(business).show()



+----------------+--------------+
|duplicate_groups|duplicate_rows|
+----------------+--------------+
|               0|          NULL|
+----------------+--------------+



In [16]:
# duplicate_count ya confirma que no hay filas completas repetidas; compruebo también que business_id (la clave) no se repite
business.groupBy("business_id").count().filter(F.col("count") > 1).show()

+-----------+-----+
|business_id|count|
+-----------+-----+
+-----------+-----+



Una fila por business_id

In [17]:
from common.profiling import missing_values_profile

missing_markers = {
    "na_count": "NA",
    "unknown_count": "Unknown",
    "empty_count": "",
}

# missing_values_profile sobre business: cuento NULLs y marcadores de texto tipo "NA"/"Unknown"/"" por columna
missing_values_profile(business, string_markers = missing_markers, output ='percentage' ).show()

+-----------+----------+---------+--------+-------------+-----------+-------------+
|column_name|null_count|nan_count|na_count|unknown_count|empty_count|missing_total|
+-----------+----------+---------+--------+-------------+-----------+-------------+
|      hours|     15.45|      0.0|     0.0|          0.0|        0.0|        15.45|
| attributes|      9.14|      0.0|     0.0|          0.0|        0.0|         9.14|
|    address|       0.0|      0.0|     0.0|          0.0|       3.41|         3.41|
| categories|      0.07|      0.0|     0.0|          0.0|        0.0|         0.07|
|postal_code|       0.0|      0.0|     0.0|          0.0|       0.05|         0.05|
+-----------+----------+---------+--------+-------------+-----------+-------------+



`is_open` tiene una media de 0,796, es decir, ~20% de negocios están cerrados. 
Coprobamos si la mayoría de nulos caen en negocios cerrados

In [18]:
business.filter(F.col("hours").isNull()).groupBy("is_open").count().show()
business.filter(F.col("attributes").isNull()).groupBy("is_open").count().show()

+-------+-----+
|is_open|count|
+-------+-----+
|      0| 7128|
|      1|16095|
+-------+-----+

+-------+-----+
|is_open|count|
+-------+-----+
|      1|12348|
|      0| 1396|
+-------+-----+



In [19]:
total_by_status = business.groupBy("is_open").count().withColumnRenamed("count", "total")

null_hours_by_status = business.filter(F.col("hours").isNull())\
    .groupBy("is_open").count().withColumnRenamed("count", "null_hours")

null_hours_by_status.join(total_by_status, "is_open")\
    .withColumn("pct", F.round(F.col("null_hours") / F.col("total") * 100, 2))\
    .show()

+-------+----------+------+-----+
|is_open|null_hours| total|  pct|
+-------+----------+------+-----+
|      0|      7128| 30648|23.26|
|      1|     16095|119698|13.45|
+-------+----------+------+-----+



In [20]:
null_attributes_by_status = business.filter(F.col("attributes").isNull())\
    .groupBy("is_open").count().withColumnRenamed("count", "null_attributes")

null_attributes_by_status.join(total_by_status, "is_open")\
    .withColumn("pct", F.round(F.col("null_attributes") / F.col("total") * 100, 2))\
    .show()

+-------+---------------+------+-----+
|is_open|null_attributes| total|  pct|
+-------+---------------+------+-----+
|      1|          12348|119698|10.32|
|      0|           1396| 30648| 4.55|
+-------+---------------+------+-----+



> **Observación: nulos en `hours` y `attributes` no siguen el mismo patrón**
>
> Comparando la tasa de nulos dentro de cada grupo de `is_open`:
>
> - **`hours`**: los negocios **cerrados** tienen una tasa de nulos claramente mayor
>   (**23,26%**) que los abiertos (**13,45%**) casi el doble. Tiene sentido: un negocio que
>   cierra deja de mantener actualizado su horario en Yelp.
> - **`attributes`**: el patrón es el **contrario** los negocios **abiertos** tienen más
>   nulos (**10,32%**) que los cerrados (**4,55%**). No se explica por `is_open`, así que debe
>   depender de otro factor (categoría del negocio, cuánto se molestó el dueño en rellenar el
>   perfil, antigüedad de la ficha...).
>
> Conclusión práctica: los nulos de `hours` sí están parcialmente explicados por el estado del
> negocio, pero los de `attributes` no — no hace falta (ni tendría sentido) aplicar ningún
> tratamiento especial ligado a `is_open` en la sección de Feature Engineering; los nulos en
> ambos campos se mantienen tal cual, como parte normal del dato (un negocio que no rellenó un
> atributo, no un error de carga).

In [21]:
business.columns

['address',
 'attributes',
 'business_id',
 'categories',
 'city',
 'hours',
 'is_open',
 'latitude',
 'longitude',
 'name',
 'postal_code',
 'review_count',
 'stars',
 'state']

In [22]:
from common.profiling import quantile_summary

# quantile_summary sobre stars / review_count: la cardinalidad ya apuntaba a que review_count está muy sesgado, compruebo los percentiles
quantile_summary(business, columns= ['stars', 'review_count']).show(truncate=False)

+------------+---+---+------+----+----+-----+-----+------+
|variable    |min|p25|median|p75 |p90 |p95  |p99  |p999  |
+------------+---+---+------+----+----+-----+-----+------+
|stars       |1.0|3.0|3.5   |4.5 |5.0 |5.0  |5.0  |5.0   |
|review_count|5.0|8.0|15.0  |37.0|96.0|171.0|449.0|7568.0|
+------------+---+---+------+----+----+-----+-----+------+



# 6. Feature Engineering

Aplano las cuatro estructuras anidadas del dataset — el objetivo técnico central de este proyecto. `categories` y `checkin.date` son arrays/strings de longitud variable, así que necesitan `split()` + `explode()`. `attributes` y `hours` son structs de esquema fijo (confirmado en la sección 3), así que se aplanan con notación de punto, sin explode.

In [23]:
# 1. categories: split + explode -> business_categories (business_id, category)

business_categories = business.withColumn('categories_string', F.split("categories", ", "))\
    .withColumn('category', F.explode(F.col('categories_string')))\
    .select('business_id','category')

business_categories.show(3, truncate=False)

+----------------------+----------------------------+
|business_id           |category                    |
+----------------------+----------------------------+
|Pns2l4eNsfO8kk83dixA6A|Doctors                     |
|Pns2l4eNsfO8kk83dixA6A|Traditional Chinese Medicine|
|Pns2l4eNsfO8kk83dixA6A|Naturopathic/Holistic       |
+----------------------+----------------------------+
only showing top 3 rows


In [24]:
# 2. attributes: struct de esquema fijo (confirmado en la sección 3) -> notación de punto, sin explode
# Cojo todos los subcampos de golpe y cuento cuántos atributos tiene rellenos cada negocio,
# para poder quedarme solo con los que aportan algo (filled_attributes_count > 0)

from functools import reduce

business_attributes = business.select("business_id", "attributes.*")

# reduce(lambda a, b: a + b, [...]) coge una lista y va combinando sus elementos
attr_cols = [c for c in business_attributes.columns if c != "business_id"]
filled_attr_cols = reduce(lambda a, b: a + b, [F.col(c).isNotNull().cast("int") for c in attr_cols])

business_attributes = business_attributes.withColumn("filled_attributes_count", filled_attr_cols)
business_attributes_clean = business_attributes.filter(F.col("filled_attributes_count") > 0)

business_attributes_clean.show(3, truncate=False)

+----------------------+----------------+-----------+-------+--------+----+-----------+----------+-----------+----------------------+--------------------------+-----------------------------------------------------------------------------------+-----------------+------+---------+-------+-------------------+-----------+---------+--------------+-----------+-----------+-----------------+---------+-----+-----+----------+-----------+--------------+-----------------+-------------------------+-------------------+------------------------+----------------------+-----------------------+-----------------------+------------------+-------+--------------------+-----+-----------------------+
|business_id           |AcceptsInsurance|AgesAllowed|Alcohol|Ambience|BYOB|BYOBCorkage|BestNights|BikeParking|BusinessAcceptsBitcoin|BusinessAcceptsCreditCards|BusinessParking                                                                    |ByAppointmentOnly|Caters|CoatCheck|Corkage|DietaryRestrictions|DogsAllo

In [25]:
print(f"Negocios con algún atributo: {business_attributes_clean.count()}, frente a {business_attributes.count()} negocios totales.")

Negocios con algún atributo: 136602, frente a 150346 negocios totales.


In [26]:
# 3. hours: struct de esquema fijo -> notación de punto, mismo criterio que attributes
# Solo son 7 subcampos (un día de la semana cada uno), así que los cojo todos de golpe

business_hours = business.select("business_id", "hours.*")

hours_cols = [c for c in business_hours.columns if c != "business_id" ]
filled_hour_cols = reduce( lambda a, b: a + b, [F.col(c).isNotNull().cast("int") for c in hours_cols])

business_hours = business_hours.withColumn("filled_hours_count", filled_hour_cols )
business_hours_clean = business_hours.filter(F.col("filled_hours_count") > 0)

business_hours_clean.show(3,truncate=False)

+----------------------+---------+--------+--------+--------+---------+---------+---------+------------------+
|business_id           |Friday   |Monday  |Saturday|Sunday  |Thursday |Tuesday  |Wednesday|filled_hours_count|
+----------------------+---------+--------+--------+--------+---------+---------+---------+------------------+
|mpf3x-BjTdTEA3yCZrAYPw|8:0-18:30|0:0-0:0 |8:0-14:0|NULL    |8:0-18:30|8:0-18:30|8:0-18:30|6                 |
|tUFrWirKiKi_TAnsVWINQQ|8:0-23:0 |8:0-22:0|8:0-23:0|8:0-22:0|8:0-22:0 |8:0-22:0 |8:0-22:0 |7                 |
|MTSW4McQd7CbVtyjqoe9mw|7:0-21:0 |7:0-20:0|7:0-21:0|7:0-21:0|7:0-20:0 |7:0-20:0 |7:0-20:0 |7                 |
+----------------------+---------+--------+--------+--------+---------+---------+---------+------------------+
only showing top 3 rows


In [27]:
print(f"Negocios con alguna hora de apertura indicada: {business_hours_clean.count()}, frente a {business_hours.count()} negocios totales.")

Negocios con alguna hora de apertura indicada: 127123, frente a 150346 negocios totales.


In [28]:
# 4. checkin.date: split + explode + to_timestamp -> checkins (business_id, checkin_time)
checkin_flatten = checkin.withColumn("checkin_time_str", F.explode(F.split("date", ", ")))\
    .withColumn("checkin_time", F.to_timestamp("checkin_time_str"))\
    .select("business_id", "checkin_time")

checkin_flatten.show(3,truncate=False)

+----------------------+-------------------+
|business_id           |checkin_time       |
+----------------------+-------------------+
|---kPU91CF4Lq2-WlRu9Lw|2020-03-13 21:10:56|
|---kPU91CF4Lq2-WlRu9Lw|2020-06-02 22:18:06|
|---kPU91CF4Lq2-WlRu9Lw|2020-07-24 22:42:27|
+----------------------+-------------------+
only showing top 3 rows


# 7. Business Questions

Cada pregunta usa las tablas aplanadas de la sección 6 (`business_categories`, `business_attributes_clean`, `business_hours_clean`, `checkin_flatten`), no el `business`/`checkin` original sin tocar.

#### Categorías de negocio

- ¿Cuáles son las categorías de negocio más comunes?
- ¿Cuántas categorías tiene un negocio en promedio? ¿Cuántos negocios tienen más de una?
- ¿Qué categorías tienen la valoración media (`stars`) más alta?

In [29]:
# ¿Cuáles son las categorías de negocio más comunes?
category_count = business_categories.groupBy("category")\
    .agg( 
        F.count("*").alias("count")
    )\
    .orderBy(F.desc("count"))

category_count.show(10, truncate= False)

+----------------+-----+
|category        |count|
+----------------+-----+
|Restaurants     |52268|
|Food            |27781|
|Shopping        |24395|
|Home Services   |14356|
|Beauty & Spas   |14292|
|Nightlife       |12281|
|Health & Medical|11890|
|Local Services  |11198|
|Bars            |11065|
|Automotive      |10773|
+----------------+-----+
only showing top 10 rows


In [30]:
# ¿Cuántas categorías tiene un negocio en promedio? ¿Cuántos negocios tienen más de una?

business_categories_recount = business_categories.groupBy("business_id")\
    .agg( 
        F.count('*').alias('count')
    )
    
business_categories_avg = business_categories_recount.agg(
        F.round( F.mean("count"), 2).alias("avg_categories")
    )

business_categories_avg.show()

+--------------+
|avg_categories|
+--------------+
|          4.45|
+--------------+



In [31]:
business_more_than_one_category = business_categories_recount.filter(F.col('count')>1)\
    .count()

print(f""" Existen {business_more_than_one_category} negocios con más de 1 categoría sobre un total de {business_categories_recount.count()} negocios. """)

 Existen 149862 negocios con más de 1 categoría sobre un total de 150243 negocios. 


In [32]:
# ¿Qué categorías tienen la valoración media (stars) más alta?
business_categories_rates = business_categories.join(business.select("business_id", "stars"), ["business_id"], "inner")
business_categories_rates.show(3)

+--------------------+--------------------+-----+
|         business_id|            category|stars|
+--------------------+--------------------+-----+
|Pns2l4eNsfO8kk83d...|             Doctors|  5.0|
|Pns2l4eNsfO8kk83d...|Traditional Chine...|  5.0|
|Pns2l4eNsfO8kk83d...|Naturopathic/Holi...|  5.0|
+--------------------+--------------------+-----+
only showing top 3 rows


In [33]:
category_stats = business_categories_rates.groupBy("category").agg(
    F.round(F.mean("stars"), 2).alias("stars_avg"),
    F.count("*").alias("count")
)

for threshold in [10, 25, 50, 100]:
    n = category_stats.filter((F.col("count") > threshold) & (F.col("stars_avg") > 4.5) ).count()
    print(f"Categorías con más de {threshold} negocios: {n}")

Categorías con más de 10 negocios: 52
Categorías con más de 25 negocios: 35
Categorías con más de 50 negocios: 21
Categorías con más de 100 negocios: 14


In [34]:
# Filtramo categorías con al menos 10 negocios para obtener aquellas con mayor valoración
highest_categories = category_stats.filter(F.col("count")>10)\
    .orderBy(F.desc("stars_avg"))

highest_categories.show(5, truncate= False)
    

+-----------------------+---------+-----+
|category               |stars_avg|count|
+-----------------------+---------+-----+
|Real Estate Photography|4.91     |32   |
|Art Tours              |4.86     |18   |
|Boudoir Photography    |4.84     |37   |
|Qi Gong                |4.83     |12   |
|Commissioned Artists   |4.81     |13   |
+-----------------------+---------+-----+
only showing top 5 rows


#### Atributos de negocio

- ¿Qué atributos son los más comunes entre los negocios?
- ¿Los negocios con un atributo concreto (p. ej. `RestaurantsDelivery`) tienen mejor valoración
  media que los que no lo tienen?

In [35]:
# ¿Qué atributos son los más comunes entre los negocios?
attr_cols = [ c for c in business_attributes_clean.columns if c not in ("business_id", "filled_attributes_count") ]
non_null_counts = business_attributes_clean.agg(
    *[F.count(c).alias(c) for c in attr_cols]
)

n = len(attr_cols)
stack_pairs = ", ".join([f"'{c}', `{c}`" for c in attr_cols])
stack_expr = f"stack({n}, {stack_pairs}) as (attribute, non_null_count)"

attribute_popularity = non_null_counts.selectExpr(stack_expr).orderBy(F.desc("non_null_count"))
attribute_popularity.show(10, truncate=False)

+--------------------------+--------------+
|attribute                 |non_null_count|
+--------------------------+--------------+
|BusinessAcceptsCreditCards|119765        |
|BusinessParking           |91085         |
|RestaurantsPriceRange2    |85314         |
|BikeParking               |72638         |
|RestaurantsTakeOut        |59857         |
|WiFi                      |56914         |
|RestaurantsDelivery       |56282         |
|GoodForKids               |53375         |
|OutdoorSeating            |48802         |
|RestaurantsReservations   |45247         |
+--------------------------+--------------+
only showing top 10 rows


In [36]:
# ¿Los negocios con un atributo concreto tienen mejor valoración media que los que no lo tienen?

business_attributes_rates = business_attributes.join(
    business.select("business_id", "stars"),
    "business_id",
    "inner"
)

# Filtramos para un atributo concreto 

business_attributes_rates.groupBy("BusinessAcceptsCreditCards").agg(
    F.avg("stars").alias("avg_stars"),
    F.count("*").alias("count")
).show()

+--------------------------+------------------+------+
|BusinessAcceptsCreditCards|         avg_stars| count|
+--------------------------+------------------+------+
|                     False|3.8404979253112033|  6025|
|                      None| 3.335616438356164|    73|
|                      NULL|  3.49421209247572| 30581|
|                      True|3.6115495262477237|113667|
+--------------------------+------------------+------+



> **Observación: `BusinessAcceptsCreditCards` y valoración media**
>
> Los negocios que no aceptan tarjeta (`False`) tienen una valoración media más alta (**3.84**)
> que los que sí la aceptan (`True`, **3.61**) — al contrario de lo que se podría esperar
> intuitivamente (que un negocio "más equipado"/moderno estaría mejor valorado). La diferencia no
> es enorme (0.23 estrellas) pero es consistente con un tamaño de muestra que la respalda de sobra
> (6.025 y 113.667 negocios en cada grupo, respectivamente).
>
> Detalle de calidad de datos que vale la pena anotar aparte: hay **73 negocios** con el valor
> literal `"None"` como **string**, distinto del `NULL` real (30.581 negocios que simplemente no
> especificaron el atributo). Es un resto de cómo se generó el dataset original (probablemente un
> `None` de Python serializado como texto en vez de quedar como nulo JSON de verdad).

#### Horarios

- ¿Cuántos negocios abren cada día de la semana?

In [37]:
# ¿Cuántos negocios abren todos los días de la semana?

all_days_open = business_hours_clean.filter(F.col("filled_hours_count")=='7').count()

print(f"Abren todos los días de la semana {all_days_open} negocios frente a {business_hours.count()} negocios totales.")

Abren todos los días de la semana 71286 negocios frente a 150346 negocios totales.


#### Negocios y reseñas

- ¿Cómo se distribuyen los negocios por ciudad/estado?
- ¿Qué negocios tienen más reseñas?
- ¿Cómo se distribuyen las puntuaciones (`stars`)?

In [38]:
# ¿Cómo se distribuyen los negocios por ciudad/estado?

from pyspark.sql.window import Window

w_total = Window.partitionBy()

business.groupBy("state").count()\
    .withColumn("pct", F.round(F.col("count") / F.sum("count").over(w_total) * 100, 2))\
    .orderBy(F.desc("count"))\
    .show(27)


+-----+-----+-----+
|state|count|  pct|
+-----+-----+-----+
|   PA|34039|22.64|
|   FL|26330|17.51|
|   TN|12056| 8.02|
|   IN|11247| 7.48|
|   MO|10913| 7.26|
|   LA| 9924|  6.6|
|   AZ| 9912| 6.59|
|   NJ| 8536| 5.68|
|   NV| 7715| 5.13|
|   AB| 5573| 3.71|
|   CA| 5203| 3.46|
|   ID| 4467| 2.97|
|   DE| 2265| 1.51|
|   IL| 2145| 1.43|
|   TX|    4|  0.0|
|   CO|    3|  0.0|
|   WA|    2|  0.0|
|   HI|    2|  0.0|
|   MA|    2|  0.0|
|   NC|    1|  0.0|
|   UT|    1|  0.0|
|   MI|    1|  0.0|
|   MT|    1|  0.0|
|   SD|    1|  0.0|
|  XMS|    1|  0.0|
|   VI|    1|  0.0|
|   VT|    1|  0.0|
+-----+-----+-----+



In [39]:
# ¿Qué negocios tienen más reseñas?
w_total = Window.partitionBy() 

business.withColumn("review_pct", F.round(F.col("review_count") / F.sum("review_count").over(w_total) * 100 , 2))\
    .select("business_id","name", "review_count" , "review_pct")\
    .orderBy(F.desc("review_count")).show(10, truncate= False)

+----------------------+----------------------------------+------------+----------+
|business_id           |name                              |review_count|review_pct|
+----------------------+----------------------------------+------------+----------+
|_ab50qdWOk0DdB6XOrBitw|Acme Oyster House                 |7568        |0.11      |
|ac1AeYqs8Z4_e2X5M3if2A|Oceana Grill                      |7400        |0.11      |
|GXFMD0Z4jEVZBCsbPf4CTQ|Hattie B’s Hot Chicken - Nashville|6093        |0.09      |
|ytynqOUb3hjKeJfRj5Tshw|Reading Terminal Market           |5721        |0.08      |
|oBNrLz4EDhiscSlbOl8uAw|Ruby Slipper - New Orleans        |5193        |0.08      |
|iSRTaT9WngzB8JJ2YKJUig|Mother's Restaurant               |5185        |0.08      |
|VQcCL9PiNL_wkGf-uF3fjg|Royal House                       |5070        |0.08      |
|_C7QiQQc47AOEv4PE3Kong|Commander's Palace                |4876        |0.07      |
|GBTPC53ZrG1ZBY3DT8Mbcw|Luke                              |4554        |0.07

In [40]:
# ¿Cómo se distribuyen las puntuaciones (stars)?

business.groupBy("stars").count()\
    .withColumn("pct", F.round( F.col("count") / F.sum("count").over(w_total) * 100 , 2))\
    .orderBy(F.desc("stars"))\
    .show()


+-----+-----+-----+
|stars|count|  pct|
+-----+-----+-----+
|  5.0|16307|10.85|
|  4.5|27181|18.08|
|  4.0|31125| 20.7|
|  3.5|26519|17.64|
|  3.0|18453|12.27|
|  2.5|14316| 9.52|
|  2.0| 9527| 6.34|
|  1.5| 4932| 3.28|
|  1.0| 1986| 1.32|
+-----+-----+-----+



#### Check-ins

- ¿Qué negocios tienen más check-ins?
- ¿Cómo se distribuyen los check-ins por hora del día o día de la semana?

In [41]:
checkin_flatten.show(1)

+--------------------+-------------------+
|         business_id|       checkin_time|
+--------------------+-------------------+
|---kPU91CF4Lq2-Wl...|2020-03-13 21:10:56|
+--------------------+-------------------+
only showing top 1 row


In [42]:
# ¿Qué negocios tienen más check-ins?

checkin_flatten.groupBy("business_id").count()\
    .orderBy(F.desc("count"))\
    .show(5, truncate= False)

+----------------------+-----+
|business_id           |count|
+----------------------+-----+
|-QI8Qi8XWH3D8y8ethnajA|52144|
|FEXhWNCMkv22qG04E83Qjg|40109|
|Eb1XmmLWyt_way5NNZ7-Pw|37562|
|c_4c5rJECZSfNgFj7frwHQ|37518|
|4i4kmYm9wgSNyF1b6gKphg|31168|
+----------------------+-----+
only showing top 5 rows


In [ ]:
# ¿Cómo se distribuyen los check-ins por hora del día o día de la semana?

checkin_flatten2 = checkin_flatten.withColumn("day_of_week", F.date_format("checkin_time", "E"))\
    .withColumn("day_of_week_num", ((F.dayofweek("checkin_time") + 5) % 7) + 1)\
    .withColumn("hour", F.hour("checkin_time"))

hour_distribution = checkin_flatten2.groupBy('hour').count()\
    .withColumn("pct", F.round( F.col("count") / F.sum("count").over(w_total) * 100, 2))\
    .orderBy(F.asc("hour"))

weekday_distribution = checkin_flatten2.groupBy("day_of_week", "day_of_week_num").count()\
    .withColumn("pct", F.round( F.col("count") / F.sum("count").over(w_total) * 100, 2))\
    .orderBy("day_of_week_num")


hour_distribution_wide = hour_distribution.groupBy().pivot("hour").agg(
    F.first("count").alias("count"),
    F.first("pct").alias("pct")
)
hour_distribution_wide.show(truncate=False, vertical=True)

weekday_distribution.show()

-RECORD 0-----------
 0_count  | 1155092 
 0_pct    | 8.65    
 1_count  | 935985  
 1_pct    | 7.01    
 2_count  | 669574  
 2_pct    | 5.01    
 3_count  | 437035  
 3_pct    | 3.27    
 4_count  | 264905  
 4_pct    | 1.98    
 5_count  | 152476  
 5_pct    | 1.14    
 6_count  | 85066   
 6_pct    | 0.64    
 7_count  | 52295   
 7_pct    | 0.39    
 8_count  | 35589   
 8_pct    | 0.27    
 9_count  | 37079   
 9_pct    | 0.28    
 10_count | 63824   
 10_pct   | 0.48    
 11_count | 115876  
 11_pct   | 0.87    
 12_count | 201427  
 12_pct   | 1.51    
 13_count | 296364  
 13_pct   | 2.22    
 14_count | 407969  
 14_pct   | 3.05    
 15_count | 587904  
 15_pct   | 4.4     
 16_count | 873108  
 16_pct   | 6.54    
 17_count | 1018438 
 17_pct   | 7.62    
 18_count | 995358  
 18_pct   | 7.45    
 19_count | 922177  
 19_pct   | 6.9     
 20_count | 858803  
 20_pct   | 6.43    
 21_count | 910653  
 21_pct   | 6.82    
 22_count | 1073857 
 22_pct   | 8.04    
 23_count | 1

+-----------+---------------+-------+-----+
|day_of_week|day_of_week_num|  count|  pct|
+-----------+---------------+-------+-----+
|        Mon|              1|1491993|11.17|
|        Tue|              2|1460432|10.93|
|        Wed|              3|1541769|11.54|
|        Thu|              4|1612496|12.07|
|        Fri|              5|1959015|14.67|
|        Sat|              6|2810469|21.04|
|        Sun|              7|2480701|18.57|
+-----------+---------------+-------+-----+



26/09/07 22:31:11 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(Blo

> **Observación: distribución de check-ins por día y por hora**
>
> **Por día de la semana**: los check-ins se concentran en fin de semana. **Sábado (21,04%)** y
> **domingo (18,57%)** juntos suman casi el **40%** del total, frente a un reparto plano entre
> semana (11-12% cada día laborable), con el **viernes (14,67%)** ya despuntando como transición
> al fin de semana.
>
> **Por hora del día**: la hora es una variable cíclica, así que hay que leerla dando la vuelta a
> la medianoche. El pico real está entre las **22h y la 1h (7-9% cada hora)** — ocio nocturno,
> cena, salir de fiesta — y el mínimo se da en plena mañana laboral, **8h-9h (≈0,27-0,28%)**, con
> una transición suave y continua entre ambos extremos a lo largo del día. Patrón coherente con
> comportamiento real de visita a negocios, sin anomalías que investigar.